# Insightia — Notebook 01 (Bloc 1 propre et validé)

Objectif, sorties et règles documentées.

## 1. Imports & chemins

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json


DATA_PATH = Path("data/commentaires_assurance_auto.csv")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
PARQUET = OUT_DIR / "comments_clean.parquet"

DATA_PATH, PARQUET, OUT_DIR


## 2. Chargement des données (CSV sans en-tête)

In [ ]:

COLS = [
    "id","date","device","canal","sentiment","motif","contexte",
    "anciennete","formule","note","urgence","region","support","commentaire"
]

if PARQUET.exists():
    df = pd.read_parquet(PARQUET)
else:
    df = pd.read_csv(DATA_PATH, sep=";", header=None, names=COLS, encoding="utf-8")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["note"] = pd.to_numeric(df["note"], errors="coerce")
df["urgence"] = pd.to_numeric(df["urgence"], errors="coerce")

df.head(3)


## 3. Contrôles minimum

In [ ]:

print("lignes:", len(df))
print("colonnes:", list(df.columns))
print("commentaires vides:", (df["commentaire"].astype(str).str.strip()=="").sum())


## 4. Bloc 1 — Lecture globale

In [ ]:

by_sentiment = df["sentiment"].value_counts(dropna=False)
by_canal = df["canal"].value_counts(dropna=False)

display(by_sentiment.to_frame("count"))
display(by_canal.to_frame("count"))

summary = {
    "n_comments": int(len(df)),
    "by_sentiment": by_sentiment.to_dict(),
    "by_canal": by_canal.to_dict(),
}
summary


### Évolution mensuelle

In [ ]:

tmp = df.dropna(subset=["date"]).copy()
tmp["month"] = tmp["date"].dt.to_period("M").astype(str)
by_month = tmp.groupby("month").size().reset_index(name="count").sort_values("month")
display(by_month)

by_month.to_csv(OUT_DIR / "volume_par_mois.csv", index=False)

plt.figure(figsize=(10,4))
plt.plot(by_month["month"], by_month["count"])
plt.xticks(rotation=45, ha="right")
plt.title("Volume de commentaires par mois")
plt.tight_layout()
plt.show()


## 5. Préparation texte (normalisation + tokenizers)

In [ ]:

import re, unicodedata

def strip_accents(s):
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if unicodedata.category(ch) != 'Mn')

def normalize_text(s):
    s = str(s).lower().strip()
    s = strip_accents(s)
    s = s.replace("’","'").replace("'","")
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

STOPWORDS_LIGHT = {"le","la","les","un","une","des","de","du","au","aux","et","ou","mais","donc","or",
                   "je","tu","il","elle","on","nous","vous","ils","elles","dans","sur","pour","par","avec","sans",
                   "ce","cet","cette","ces","qui","que","quoi","dont"}

STOPWORDS_STRUCT = {"cest","pas","jai","suis","etre","avoir","tout","rien","tres","trop","encore","toujours",
                    "juste","peux","peut","fait","faire","vais","aller","merci","bonjour","soir","jour","fois",
                    "quand","meme","alors","est"}

def tokenize_light(s):
    toks = normalize_text(s).split()
    return [t for t in toks if len(t)>=3 and t not in STOPWORDS_LIGHT and not t.isdigit()]

def tokenize_clean(s):
    toks = normalize_text(s).split()
    return [t for t in toks if len(t)>=3 and t not in STOPWORDS_LIGHT and t not in STOPWORDS_STRUCT and not t.isdigit()]


## 6. Nuage de mots — brut

In [ ]:

from wordcloud import WordCloud

tokens_raw = []
for s in df["commentaire"].astype(str):
    tokens_raw.extend(tokenize_light(s))

wc_raw = WordCloud(width=1400, height=800, background_color="white", collocations=False)    .generate(" ".join(tokens_raw))

plt.figure(figsize=(12,6))
plt.imshow(wc_raw, interpolation="bilinear")
plt.axis("off")
plt.title("Nuage de mots — brut")
plt.show()

wc_raw.to_file(OUT_DIR / "wordcloud_raw.png")


## 7. Nuage de mots — nettoyé

In [ ]:

tokens_clean = []
for s in df["commentaire"].astype(str):
    tokens_clean.extend(tokenize_clean(s))

wc_clean = WordCloud(width=1400, height=800, background_color="white", collocations=False)    .generate(" ".join(tokens_clean))

plt.figure(figsize=(12,6))
plt.imshow(wc_clean, interpolation="bilinear")
plt.axis("off")
plt.title("Nuage de mots — nettoyé")
plt.show()

wc_clean.to_file(OUT_DIR / "wordcloud_clean.png")


## 8. Relations de mots — nettoyées

In [ ]:

from collections import Counter
from itertools import combinations

top_k_words = 60
min_pair_count = 40

word_counts = Counter()
docs = []

for s in df["commentaire"].astype(str):
    toks = list(dict.fromkeys(tokenize_clean(s)))
    docs.append(toks)
    word_counts.update(toks)

vocab = set([w for w,_ in word_counts.most_common(top_k_words)])
pairs = Counter()

for toks in docs:
    toks = [t for t in toks if t in vocab]
    for a,b in combinations(sorted(set(toks)),2):
        pairs[(a,b)] += 1

edges = pd.DataFrame(
    [{"word_a":a,"word_b":b,"count":c} for (a,b),c in pairs.items() if c>=min_pair_count]
).sort_values("count", ascending=False)

edges.head(20)


### Export relations + résumé

In [ ]:

edges.to_csv(OUT_DIR / "word_relations_edges_clean.csv", index=False)

summary["params"] = {
    "tokenizer": "tokenize_clean",
    "top_k_words": top_k_words,
    "min_pair_count": min_pair_count
}
summary["by_month"] = dict(zip(by_month["month"], by_month["count"]))

with open(OUT_DIR / "block1_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

(OUT_DIR / "word_relations_edges_clean.csv", OUT_DIR / "block1_summary.json")


✅ Notebook 01 validé.